# Урок 30 — Python Networking as a System

## Sockets, TCP, Blocking I/O, Concurrency та Event Loop

**Модуль 4 · Network & Concurrent Systems · Автор: Viktor Nikoriak**

---

# Про що цей урок?

До цього моменту ми працювали переважно всередині одного процесу.

Наш код виглядав приблизно так:

```python
x = load_data()
y = process(x)
save(y)
```

Все було локальним:

* пам’ять
* змінні
* функції
* структури даних

І головне:

> процес повністю контролював execution flow.

Але networking повністю змінює цю модель.

Тому що тепер твоя програма починає взаємодіяти:

* з операційною системою
* мережею
* іншими процесами
* фізичним обладнанням
* затримками
* packet loss
* concurrency
* scheduler'ом ядра

І тут відбувається головний зсув мислення:

> networking — це НЕ “обмін повідомленнями”

Насправді networking — це:

* blocking execution
* synchronization
* kernel buffers
* TCP state machine
* interrupts
* asynchronous events
* coordination між процесами

Більшість початківців бачать networking так:

```text
Client
   ↓
Message
   ↓
Server
```

Але реальна система виглядає значно глибше:

```text
┌──────────────────────────────┐
│      Python User Space       │
│                              │
│  send() / recv() / accept()  │
└─────────────┬────────────────┘
              │ syscall
══════════════╪══════════════════════
              ▼
┌──────────────────────────────┐
│         OS KERNEL            │
│                              │
│  TCP Stack                   │
│  Socket Buffers              │
│  Scheduler                   │
│  Interrupt Handlers          │
│  File Descriptor Table       │
└─────────────┬────────────────┘
              │ NIC Driver
══════════════╪══════════════════════
              ▼
┌──────────────────────────────┐
│     Network Interface Card   │
└─────────────┬────────────────┘
              │ electrical signal
══════════════╪══════════════════════
              ▼
        Physical Network
```

І саме тут народжується systems thinking.

Тому що тепер:

* процес може “заснути”
* recv() може заблокувати thread
* scheduler може забрати CPU
* kernel може тримати дані в socket buffer
* event loop може зупинитися через один blocking call
* packets можуть приходити частинами
* server може deadlock'нути

У цьому модулі ми більше НЕ будемо мислити як:

* “програмісти функцій”

Ми почнемо мислити як:

* systems engineers
* backend architects
* network engineers
* distributed systems developers

---

# Головна мета модуля

Навчитися МЕНТАЛЬНО симулювати execution flow мережевої системи.

Не просто писати:

```python
sock.recv()
```

А розуміти:

* що робить kernel
* де process блокується
* хто володіє CPU
* як ОС будить process
* як TCP збирає byte stream
* як працює concurrency
* чому server “зависає”
* чому asyncio може втратити весь parallelism
* чому networking змінює архітектуру програм

---

# У цьому модулі ми перейдемо

* від sequential execution → до event-driven systems
* від локального коду → до distributed communication
* від простих функцій → до asynchronous architecture
* від “дані в пам'яті” → до data streams
* від control flow → до coordination flow

---

# Структура уроку

| Частина | Тема                                  |
| ------- | ------------------------------------- |
| 1       | Що таке socket насправді?             |
| 2       | Python Process vs. OS Kernel          |
| 3       | TCP як byte stream                    |
| 4       | recv() і blocking execution           |
| 5       | Socket Buffers та movement of data    |
| 6       | TCP Handshake та connection lifecycle |
| 7       | EOF, FIN та disconnect                |
| 8       | Partial Messages та framing problem   |
| 9       | Deadlocks та buffered I/O             |
| 10      | Threading та I/O concurrency          |
| 11      | GIL — міфи та реальність              |
| 12      | Event Loop та asyncio                 |
| 13      | Blocking calls inside async systems   |
| 14      | select(), epoll(), multiplexing       |
| 15      | Архітектура production servers        |
| 16      | Debugging networking systems          |

---

# Що особливо важливо в цьому модулі?

У цьому модулі НЕ можна:

* просто читати код
* механічно запам’ятовувати API
* думати лінійно

Тому що networking — це:

> взаємодія багатьох незалежних систем у часі.

І хороший network engineer:

* не читає код “по рядках”
* а симулює execution flow всієї системи:

  * kernel
  * scheduler
  * buffers
  * threads
  * packets
  * event loop
  * timing
  * synchronization

Саме це і є справжнє системне мислення.



# 1. Що таке socket насправді?

Коли початківці вперше бачать networking code, вони зазвичай уявляють це так:

```text
Client
   ↓
Message
   ↓
Server
````

І це створює дуже небезпечну ілюзію.

Тому що в реальності socket — це НЕ:

* “канал повідомлень”
* “чат між двома Python програмами”
* “магічна труба”

Socket — це набагато нижчорівневіша концепція.

---

# Socket = Interface між процесом та OS Kernel

Python НЕ працює з мережею напряму.

Python працює лише з:

* пам’яттю процесу
* системними викликами (syscalls)

А вся реальна мережева магія відбувається всередині:

```text
OS Kernel
```

Саме ядро ОС:

* керує TCP/IP stack
* приймає packets
* збирає byte stream
* керує retransmission
* перевіряє checksums
* тримає socket buffers
* будить процеси
* планує execution через scheduler

Socket — це програмний endpoint,
через який Python process взаємодіє з kernel networking stack.

---

# Реальна архітектура networking

```text
┌──────────────────────────────┐
│      Python Process          │
│                              │
│ send() recv() accept()       │
└─────────────┬────────────────┘
              │ syscall
══════════════╪══════════════════════
              ▼
┌──────────────────────────────┐
│         OS KERNEL            │
│                              │
│ TCP/IP Stack                 │
│ Socket Buffers               │
│ Scheduler                    │
│ Interrupt Handlers           │
│ File Descriptor Table        │
└─────────────┬────────────────┘
              │ driver
══════════════╪══════════════════════
              ▼
┌──────────────────────────────┐
│  NIC (Network Card)          │
└─────────────┬────────────────┘
              │ electrical signal
══════════════╪══════════════════════
              ▼
        Physical Network
```

---

# Критичний mental shift

Початківець думає:

```python
data = sock.recv(1024)
```

означає:

```text
"отримати повідомлення від іншої програми"
```

Але реальність інша.

recv() означає:

```text
"запитати kernel,
чи є bytes у socket receive buffer"
```

І якщо bytes ще не прийшли:

* процес засинає
* scheduler забирає CPU
* execution flow зупиняється

Networking — це не “виклик функцій”.

Це:

* coordination
* waiting
* synchronization
* asynchronous physical events

---

# Socket як file descriptor

У Linux/Unix socket — це фактично:

```text
special file descriptor
```

Тому sockets можна:

* читати
* писати
* multiplex через select/epoll
* передавати між process
* monitor через kernel event system

Для ОС socket дуже схожий на:

* файл
* pipe
* stdin/stdout

Саме тому networking у Unix настільки потужний:

> "everything is a file"

---

# Telephone Switchboard Mental Model

Більшість новачків думають,
що server socket — це “одна труба для всіх клієнтів”.

Це неправильно.

Listening socket — це:

```text
рецепція готелю
```

Його задача:

* стояти на порту
* чекати нові connection requests

Коли клієнт підключається:

```python
conn, addr = server.accept()
```

kernel:

* створює НОВИЙ socket
* виділяє нові buffers
* створює окремий connection state

А listening socket:

* НЕ спілкується з клієнтом
* він лише приймає нові connections

І одразу повертається чекати наступних.

---

# Що особливо важливо зрозуміти?

TCP НЕ працює повідомленнями.

TCP = continuous byte stream.

Це означає:

```python
sock.send(b"Hello")
sock.send(b"World")
```

НЕ гарантує:

```python
recv() -> b"Hello"
recv() -> b"World"
```

Kernel може:

* об’єднати packets
* розбити packets
* затримати packets

recv() читає:

> "все що зараз лежить у receive buffer"

Саме тому production protocols:

* HTTP
* WebSocket
* PostgreSQL protocol
* Redis protocol

мають власний message framing layer.

---

# Blocking — головна проблема networking

CPU працює у мільйони разів швидше за мережу.

Тому майже кожна network operation:

* accept()
* recv()
* connect()

це:

```text
WAITING OPERATION
```

Коли процес виконує:

```python
data = sock.recv(1024)
```

і buffer порожній:

kernel:

* блокує process
* scheduler забирає CPU
* thread переходить у sleep state

І це фундаментальна причина появи:

* threading
* multiprocessing
* asyncio
* event loops
* epoll/select
* reactive systems
* actor model architecture

Networking буквально змушує програміста мислити системно.

---

# Головна ідея цього модуля

У цьому модулі ми НЕ будемо:

* “запам’ятовувати socket API”

Ми будемо:

* ментально симулювати execution flow
* бачити kernel behavior
* розуміти blocking
* бачити scheduler interaction
* розуміти socket lifecycle
* мислити як systems engineers
____

# 1.1. Socket у Python: мінімальний експеримент

Тепер ми не просто читаємо визначення socket.

Ми створимо маленьку систему:

```text
Client Process
     ↓
OS Kernel
     ↓
TCP Socket Buffer
     ↓
Server Process
```

Мета експерименту:

- побачити, що socket — це endpoint
- побачити різницю між listening socket і connection socket
- побачити, де програма блокується
- побачити, що TCP передає bytes, а не Python strings
- побачити базовий lifecycle: socket → bind → listen → accept → recv → send → close

In [ ]:
import socket
import threading
import time


## 1.2. Server socket

Server socket має спеціальну роль.

Він НЕ читає дані від клієнта напряму.

Його задача:

```text
1. створити endpoint
2. прив'язати endpoint до IP + PORT
3. сказати OS Kernel: "я готовий приймати connection requests"
4. чекати connect()
```

Критично важливо:

```python
server_socket.accept()
```

повертає НОВИЙ socket:

```python
conn, addr = server_socket.accept()
```

- `server_socket` — слухає порт
- `conn` — окремий socket для конкретного клієнта

In [ ]:
HOST = "127.0.0.1"
PORT = 5050


def run_server():
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

    server_socket.bind((HOST, PORT))
    server_socket.listen(1)

    print("[SERVER] Listening on", (HOST, PORT))
    print("[SERVER] Waiting at accept()...")

    conn, addr = server_socket.accept()

    print("[SERVER] accept() returned")
    print("[SERVER] Client address:", addr)
    print("[SERVER] Connection socket:", conn)

    print("[SERVER] Waiting at recv()...")

    data = conn.recv(1024)

    print("[SERVER] recv() returned")
    print("[SERVER] Raw bytes:", data)
    print("[SERVER] Decoded text:", data.decode("utf-8"))

    response = b"Hello from server"
    conn.sendall(response)

    print("[SERVER] Response sent")

    conn.close()
    server_socket.close()

    print("[SERVER] Closed")

## 1.3. Client socket

Client socket має іншу роль.

Він не слухає порт.

Він ініціює connection:

```python
client_socket.connect((HOST, PORT))
```

У цей момент Python process просить OS Kernel:

```text
"створи TCP connection до цього IP і PORT"
```

Далі OS виконує TCP handshake.

Python не керує handshake вручну.
Це робить kernel.

In [ ]:
def run_client():
    time.sleep(1)

    client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    print("[CLIENT] Connecting...")

    client_socket.connect((HOST, PORT))

    print("[CLIENT] connect() returned")

    message = "Hello from client"
    encoded_message = message.encode("utf-8")

    print("[CLIENT] Sending bytes:", encoded_message)

    client_socket.sendall(encoded_message)

    print("[CLIENT] Waiting for response...")

    response = client_socket.recv(1024)

    print("[CLIENT] Raw response:", response)
    print("[CLIENT] Decoded response:", response.decode("utf-8"))

    client_socket.close()

    print("[CLIENT] Closed")

## 1.4. Запускаємо server і client в одному notebook

У реальній системі server і client зазвичай запускаються в різних процесах.

Але в notebook нам зручно використати `threading`, щоб симулювати дві незалежні сторони:

```text
Thread 1 → server
Thread 2 → client
```

Це НЕ означає, що networking потребує threading завжди.

Ми використовуємо threading лише для демонстрації,
щоб server і client могли працювати одночасно в одній notebook-сесії.

In [ ]:
server_thread = threading.Thread(target=run_server)
client_thread = threading.Thread(target=run_client)

server_thread.start()
client_thread.start()

server_thread.join()
client_thread.join()

# 1.5. Що тут реально відбулося?

Подивимось на execution flow:

```text
[T=0] Server створює socket
[T=1] Server bind() → OS закріплює port 5050
[T=2] Server listen() → OS створює connection queue
[T=3] Server accept() → server thread засинає

[T=4] Client створює socket
[T=5] Client connect() → OS запускає TCP handshake
[T=6] Kernel завершує handshake
[T=7] Server accept() прокидається
[T=8] Kernel створює новий connection socket

[T=9] Client sendall() → bytes копіюються в OS send buffer
[T=10] Server recv() → bytes копіюються з OS receive buffer у Python memory
[T=11] Server sendall() → відповідь іде назад
[T=12] Client recv() → читає відповідь
[T=13] close() → TCP connection завершується
```

Головна ідея:

> Python не передає дані напряму іншому Python-процесу.

Python передає bytes у OS Kernel.
Kernel керує TCP.

# 1.6. Обмеження цього прикладу

Цей приклад правильний для базового розуміння,
але він НЕ є production server.

Його обмеження:

## 1. Сервер приймає тільки одного клієнта

```python
server_socket.listen(1)
conn, addr = server_socket.accept()
```

Після одного клієнта сервер завершується.

У production потрібен цикл:

```python
while True:
    conn, addr = server_socket.accept()
```

---

## 2. recv(1024) НЕ гарантує повне повідомлення

```python
data = conn.recv(1024)
```

Це означає:

```text
"прочитати максимум 1024 bytes"
```

А НЕ:

```text
"прочитати одне повне повідомлення"
```

TCP не знає, де повідомлення починається і закінчується.

---

## 3. Немає message framing

У реальних протоколах треба явно визначити межу повідомлення:

- delimiter: `\n`
- fixed-size header
- length-prefix protocol
- JSON lines
- HTTP headers + Content-Length

---

## 4. Немає обробки disconnect

Якщо клієнт закриє connection,
то:

```python
recv()
```

поверне:

```python
b""
```

Це треба обробляти:

```python
if not data:
    break
```

---

## 5. Немає timeout

Якщо client підключився, але нічого не надсилає,
server може висіти на:

```python
conn.recv(1024)
```

дуже довго.

Production systems використовують:

```python
socket.settimeout()
```

або event loop.

---

## 6. Немає concurrency для багатьох клієнтів

Якщо один client довго обробляється,
інші clients чекають.

Для цього використовують:

- threads
- process pool
- asyncio
- select/epoll
- event-driven architecture

---

## 7. Дані мають бути bytes

Socket не приймає звичайний Python `str`.

Неправильно:

```python
sock.sendall("hello")
```

Правильно:

```python
sock.sendall("hello".encode("utf-8"))
```

Бо network передає bytes.

# 1.7. Питання для студентів

Перед тим як рухатись далі, дай відповідь:

## Question 1

Чому `accept()` повертає новий socket,
а не використовує той самий `server_socket`?

---

## Question 2

Що буде, якщо client виконає:

```python
client_socket.sendall(b"Hello")
client_socket.sendall(b"World")
```

А server виконає тільки:

```python
data = conn.recv(1024)
```

Чи гарантовано server отримає `b"Hello"`?

---

## Question 3

Чому socket працює з `bytes`,
а не з Python `str`?

---

## Question 4

Де саме програма може "зависнути"?

Варіанти:

- `bind()`
- `listen()`
- `accept()`
- `recv()`
- `sendall()`
- `close()`

---

## Question 5

Чому цей код не можна одразу використовувати як web server?

# 2. Python Process vs. OS Kernel

До цього моменту ми писали код так,
ніби Python сам усе контролює:

```python
x = read_file()
y = process(x)
print(y)
```

Але networking повністю руйнує цю модель.

Тому що socket code — це постійна взаємодія між:

```text
Python Process
        ↕
OS Kernel
```

І найважливіше:

> Python НЕ працює з мережею напряму.

Він лише:
- виконує syscall
- передає контроль kernel
- чекає результат

---

# Що таке syscall?

Коли Python виконує:

```python
sock.recv(1024)
```

це НЕ “звичайна Python функція”.

Насправді відбувається:

```text
Python
   ↓
syscall
   ↓
OS Kernel
   ↓
TCP/IP Stack
```

Тобто процес буквально каже ядру:

```text
"перевір network socket"
```

---

# User Space vs Kernel Space

Комп’ютер логічно поділений на дві великі області:

```text
┌──────────────────────────────┐
│        User Space            │
│                              │
│ Python                       │
│ VSCode                       │
│ Browser                      │
│ Games                        │
└─────────────┬────────────────┘
              │ syscalls
══════════════╪══════════════════════
              ▼
┌──────────────────────────────┐
│        Kernel Space          │
│                              │
│ Scheduler                    │
│ TCP/IP Stack                 │
│ Drivers                      │
│ Memory Manager               │
│ File System                  │
└──────────────────────────────┘
```

Python process НЕ має прямого доступу:
- до network card
- до RAM інших процесів
- до scheduler
- до hardware

Усе це контролює kernel.

---

# Що реально відбувається у recv()?

Подивимось на:

```python
data = sock.recv(1024)
```

Початківець думає:

```text
"отримати повідомлення"
```

Але execution flow значно складніший:

```text
1. Python викликає recv()
2. CPU переходить у kernel mode
3. Kernel перевіряє socket receive buffer
4. Якщо buffer empty:
       process SLEEPS
5. Scheduler забирає CPU
6. NIC отримує packet
7. Kernel копіює bytes у receive buffer
8. Scheduler будить process
9. recv() повертає bytes
```

---

# Найважливіша ідея networking

Коли process виконує:

```python
recv()
```

і даних немає —

```text
PROCESS ПЕРЕСТАЄ ВИКОНУВАТИСЬ
```

Це критичний mental shift.

Багато джунів думають:

```text
"програма просто чекає"
```

Ні.

Програма:
- literally sleeping
- не має CPU
- scheduler прибрав її з execution queue

---

# Мінімальний експеримент


In [ ]:
import socket

server = socket.socket()

server.bind(("127.0.0.1", 5051))
server.listen(1)

print("BEFORE ACCEPT")

conn, addr = server.accept()

print("AFTER ACCEPT")

Запусти цей код.

Що ти побачиш?

```text
BEFORE ACCEPT
```

І все.

Програма “зависне”.

---

# Але що означає "зависне"?

Це НЕ crash.

І НЕ infinite loop.

Процес просто перейшов у:

```text
BLOCKING STATE
```

Kernel:
- поставив process sleep
- звільнив CPU
- чекає TCP SYN packet

---

# Mental Visualization

Уяви:

```text
accept()
```

це як:

```text
стояти біля дверей
і чекати поки хтось постукає
```

Поки ніхто не стукає:
- ти нічого не робиш
- CPU тобі не потрібен
- scheduler може виконувати інші задачі

---

# Де networking "ламає" sequential thinking?

У звичайному коді:

```python
a()
b()
c()
```

ми очікуємо:

```text
a -> b -> c
```

Але networking code виглядає так:

```python
recv()
```

і execution flow стає:

```text
run
↓
sleep
↓
hardware interrupt
↓
scheduler wakeup
↓
continue execution
```

Тобто hardware починає впливати на flow програми.

---

# Чому це важливо?

Саме через blocking networking породив:

- threads
- asyncio
- event loops
- epoll/select
- reactive systems
- async frameworks
- Node.js architecture
- Nginx architecture

Бо sequential process не може ефективно чекати тисячі клієнтів.

---

# Головний висновок

Networking — це НЕ:
- "виклики функцій"

Networking — це:
- synchronization між process та kernel
- waiting for external events
- coordination через buffers
- interrupt-driven execution

І хороший backend engineer завжди мислить:

```text
Хто зараз володіє CPU?
Python process?
чи OS Kernel?
```

# 3. TCP як byte stream

Одна з найбільш небезпечних помилок у networking:

> думати що TCP працює повідомленнями.

Але перед цим потрібно зрозуміти:

```text
ЩО ТАКЕ TCP ВЗАГАЛІ?
```

---

# Що таке TCP?

TCP =

```text
Transmission Control Protocol
```

---

# Розберемо назву

## Transmission

```text
передача
```

TCP відповідає за передачу даних між двома машинами.

---

## Control

```text
контроль
```

TCP:
- перевіряє чи packets дійшли
- робить retransmission
- слідкує за порядком bytes
- контролює flow
- контролює connection state

---

## Protocol

```text
набір правил
```

TCP — це стандарт,
який описує:
- як машини встановлюють connection
- як передають bytes
- як підтверджують доставку
- як закривають connection

---

# Де TCP знаходиться?

TCP — це частина:

```text
TCP/IP Stack
```

який знаходиться всередині:

```text
OS Kernel
```

Python НЕ реалізує TCP самостійно.

Коли ти робиш:

```python
sock.sendall(data)
```

Python:
- передає bytes kernel
- а вже kernel TCP stack:
    - ділить bytes на packets
    - додає headers
    - робить retransmission
    - відправляє через NIC

---

# Що TCP гарантує?

TCP — це:

```text
reliable ordered byte stream
```

Розберемо кожне слово.

---

# Reliable

```text
надійний
```

TCP:
- перевіряє checksum
- retransmit lost packets
- підтверджує доставку через ACK

Якщо packet загубився —
TCP спробує відправити його ще раз.

---

# Ordered

```text
в правильному порядку
```

Якщо client надіслав:

```text
A B C D
```

то application отримає:

```text
A B C D
```

а НЕ:

```text
C A D B
```

TCP збирає bytes у правильному порядку.

---

# Byte Stream

І ось тут найважливіше.

TCP бачить лише:

```text
безперервний потік bytes
```

Для TCP НЕ існує:
- повідомлень
- JSON
- Python strings
- HTTP requests
- chat messages

Є лише:

```text
0 і 1
```

---

# Помилка вважати що:


```python
sock.send(b"Hello")
sock.send(b"World")
```

означає:

```text
Message 1
Message 2
```

І очікують:

```python
recv() -> b"Hello"
recv() -> b"World"
```

Але TCP так НЕ працює.

---

# Що реально бачить TCP?

TCP бачить:

```text
HelloWorld
```

як:

```text
один довгий byte stream
```

---

# Mental Model: Водопровідна труба

TCP socket найкраще уявляти як:

```text
водопровідну трубу
```

send():
- ллє воду у трубу

recv():
- зачерпує воду відром

І відро може:
- зачерпнути половину
- одну порцію
- дві порції одразу

TCP НЕ знає:
- де “Hello”
- де “World”
- де “JSON”
- де “кінець повідомлення”

---

# Чому так відбувається?

Тому що між send() і recv() є:

```text
Python Process
↓
OS Send Buffer
↓
TCP Stack
↓
NIC
↓
Physical Network
↓
Remote NIC
↓
Remote TCP Stack
↓
Remote Receive Buffer
↓
recv()
```

І TCP:
- може об’єднати packets
- ділити packets
- retransmit packets
- fragment packets

---

# Packet ≠ Message

Це critical concept.

TCP packet:
- transport-level structure

Message:
- application-level structure

TCP НЕ знає:
- що таке JSON
- що таке HTTP
- що таке рядок
- що таке chat message

TCP бачить лише:

```text
bytes
```

---

# Мінімальний експеримент


In [ ]:
# TCP DEMO
# Reliable Ordered Byte Stream
#
# Цей приклад показує:
# 1. TCP connection
# 2. TCP handshake
# 3. Byte stream behavior
# 4. recv() НЕ читає "повідомлення"
# 5. TCP reorder / buffering abstraction

import socket
import threading
import time


HOST = "127.0.0.1"
PORT = 5057


# =========================
# SERVER
# =========================

def run_server():

    server = socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM
    )

    server.setsockopt(
        socket.SOL_SOCKET,
        socket.SO_REUSEADDR,
        1
    )

    print("\n[SERVER] bind()")

    server.bind((HOST, PORT))

    print("[SERVER] listen()")

    server.listen(1)

    print("[SERVER] waiting accept()...\n")

    conn, addr = server.accept()

    print("[SERVER] accept() returned")
    print("[SERVER] client:", addr)

    # =========================
    # recv #1
    # =========================

    print("\n[SERVER] recv #1 waiting...")

    data1 = conn.recv(5)

    print("[SERVER] recv #1 returned:", data1)

    # =========================
    # recv #2
    # =========================

    print("\n[SERVER] recv #2 waiting...")

    data2 = conn.recv(1024)

    print("[SERVER] recv #2 returned:", data2)

    # =========================
    # recv #3
    # =========================

    print("\n[SERVER] recv #3 waiting EOF...")

    data3 = conn.recv(1024)

    print("[SERVER] recv #3 returned:", data3)

    if data3 == b"":
        print("\n[SERVER] EOF detected")
        print("[SERVER] client closed connection")

    conn.close()
    server.close()

    print("\n[SERVER] closed")


# =========================
# CLIENT
# =========================

def run_client():

    time.sleep(1)

    client = socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM
    )

    print("\n[CLIENT] connect()")

    client.connect((HOST, PORT))

    print("[CLIENT] TCP connection established")

    # =========================
    # SEND #1
    # =========================

    print("\n[CLIENT] sendall(b'Hello')")

    client.sendall(b"Hello")

    time.sleep(1)

    # =========================
    # SEND #2
    # =========================

    print("\n[CLIENT] sendall(b'World')")

    client.sendall(b"World")

    time.sleep(1)

    print("\n[CLIENT] close()")

    client.close()


# =========================
# THREADS
# =========================

server_thread = threading.Thread(target=run_server)

client_thread = threading.Thread(target=run_client)

server_thread.start()

client_thread.start()

server_thread.join()

client_thread.join()

Що надрукує server?

Можливі варіанти:

```python
b"HelloWorld"
```

або:

```python
b"Hello"
```

або навіть:

```python
b"Hel"
```

---

# Чому так відбувається?

Тому що recv():

```python
recv(1024)
```

означає:

```text
"дай мені все що зараз лежить
у receive buffer
до максимум 1024 bytes"
```

А НЕ:

```text
"дай одне повне повідомлення"
```

---

# Kernel buffering

Коли client виконує:

```python
sendall(b"Hello")
```

дані:
- НЕ йдуть одразу у Python server
- вони потрапляють у OS send buffer

Потім:
- TCP stack
- NIC
- network
- remote OS

І лише потім:
- receive buffer іншої машини

recv() читає:
- НЕ network
- а local kernel receive buffer

---

# Production Problem

Це породжує одну з найважливіших проблем networking:

```text
Message Framing Problem
```

TCP НЕ знає:
- де починається message
- де закінчується message

Тому application layer мусить це вирішувати.

---

# Як це вирішують real protocols?

HTTP:

```text
Content-Length
```

WebSocket:

```text
frame header
```

Redis protocol:

```text
length-prefixed protocol
```

JSON Lines:

```text
delimiter = \n
```

---

# Дуже часта помилка:

```python
data = recv(1000000)

json.loads(data)
```

Іноді працює.

Іноді:
- JSON broken
- data incomplete
- random parse errors

Чому?

Бо recv() повернув лише частину stream.

---

# Головний висновок

TCP НЕ гарантує:
- boundaries
- message chunks
- application packets

TCP гарантує лише:

```text
ordered reliable byte stream
```

# 4. recv() і blocking execution

recv() — одна з найважливіших функцій у networking.

І водночас —
одна з найбільш неправильно зрозумілих.

---
# Що реально робить recv()?

recv() означає:

```text
"перевірити receive buffer kernel
і якщо він пустий —
заблокувати process"
```

---

# Мінімальний приклад


In [ ]:
import socket

server = socket.socket()

server.bind(("127.0.0.1", 5052))
server.listen(1)

conn, addr = server.accept()

print("BEFORE RECV")

data = conn.recv(1024)

print("AFTER RECV")

Якщо client нічого не надсилає —

ти побачиш:

```text
BEFORE RECV
```

і все.

---

# Що відбулося?

recv():
- перевірив receive buffer
- buffer empty
- kernel заблокував thread

Process:
- більше НЕ виконується
- НЕ використовує CPU
- waiting for network interrupt

---

# Дуже важлива ідея

recv() НЕ "polling".

recv() НЕ крутиться у циклі.

Kernel:
- literally puts thread to sleep

---

# Timeline execution flow

```text
[T=0]
Python calls recv()

[T=1]
CPU enters kernel mode

[T=2]
Kernel checks receive buffer

[T=3]
Buffer empty

[T=4]
Process state:
BLOCKED / SLEEPING

[T=5]
Scheduler switches to another process

[T=6]
NIC receives packet

[T=7]
Hardware interrupt fires

[T=8]
Kernel copies bytes into receive buffer

[T=9]
Kernel wakes Python process

[T=10]
recv() returns bytes
```

---

# Чому це критично для architecture?

Уяви server:

```python
while True:
    conn, addr = server.accept()

    data = conn.recv(1024)

    process(data)
```

Якщо:
- Client A підключився
- але думає 20 секунд перед send()

то server:
- ЗАБЛОКОВАНИЙ
- НЕ приймає інших clients

---

# Mental Visualization

Уяви касира в магазині.

Клієнт підійшов до каси —
і 20 хвилин шукає гроші.

Поки касир чекає:
- вся черга стоїть
- нові клієнти не обробляються

Саме так працює blocking server.

---

# Як це вирішують production systems?

Є кілька стратегій:

## 1. Thread per client

```text
1 client = 1 thread
```

---

## 2. Process pool

```text
1 client = 1 process
```

---

## 3. Non-blocking sockets

```python
sock.setblocking(False)
```

---

## 4. Event loops

```text
select()
epoll()
asyncio
uvloop
```

---

# Чому asyncio взагалі існує?

Саме через blocking recv().

Ідея event loop:

```text
НЕ чекати одному socket
а обслуговувати тисячі socket
поки інші waiting
```

---

# Головний висновок

recv() — це не просто:
- "отримати bytes"

recv() — це:
- synchronization point
- scheduler interaction
- blocking operation
- control handoff to kernel

І це фундамент networking architecture.

# 5. Socket Buffers та movement of data

Коли ти робиш:

```python
sock.sendall(b"Hello")
```

bytes НЕ летять одразу:
- у Python process іншої машини
- у recv()

---

# Що реально відбувається?

Дані проходять через кілька рівнів:

```text
Python Memory
↓
OS Send Buffer
↓
TCP Stack
↓
NIC
↓
Network
↓
Remote NIC
↓
Remote TCP Stack
↓
Remote Receive Buffer
↓
recv()
```

---

# Найважливіший mental shift

recv() читає:

```text
НЕ network
```

recv() читає:

```text
local kernel receive buffer
```

---

# Що таке Socket Buffer?

Socket buffer —
це область RAM всередині OS kernel.

Є:
- Send Buffer
- Receive Buffer

Kernel використовує їх щоб:
- тимчасово тримати bytes
- керувати швидкістю
- згладжувати network latency

---

# Чому buffers потрібні?

CPU:
- дуже швидкий

Network:
- дуже повільна

Тому kernel:
- накопичує bytes
- передає їх асинхронно

---

# Важливий наслідок

sendall():

```python
sock.sendall(data)
```

зазвичай НЕ означає:

```text
"дані вже дійшли"
```

Це означає:

```text
"дані успішно скопійовані у kernel send buffer"
```

# 6. TCP Handshake та connection lifecycle

TCP — connection-oriented protocol.

Перед передачею даних TCP створює:

```text
logical connection
```

---

# 3-Way Handshake

TCP connection починається так:

```text
CLIENT                 SERVER

SYN        ─────────►

           ◄─────────    SYN-ACK

ACK        ─────────►
```

---

# Що означає SYN?

```text
Synchronize
```

Client каже:

```text
"хочу створити connection"
```

---

# Що робить SYN-ACK?

Server каже:

```text
"ок, я готовий"
```

---

# ACK

Client підтверджує:

```text
"connection established"
```

І лише після цього:

```python
send()
recv()
```

стають можливими.

---

# TCP State Machine

TCP socket постійно змінює state:

```text
CLOSED
LISTEN
SYN_SENT
ESTABLISHED
FIN_WAIT
TIME_WAIT
CLOSED
```

---

# Чому це важливо?

Бо networking —
це НЕ просто "дані".

Це:
- state machine
- synchronization
- lifecycle management

# 7. EOF, FIN та disconnect

Коли client виконує:

```python
sock.close()
```

TCP НЕ "знищує connection" миттєво.

---

# Що реально відбувається?

Kernel:
- надсилає FIN packet
- повідомляє:

```text
"я більше не надсилатиму bytes"
```

---

# Що бачить recv()?

Server:

```python
data = conn.recv(1024)
```

отримує:

```python
b""
```

---

# Це НЕ помилка

```python
b""
```

означає:

```text
EOF (End Of File)
```

або:

```text
remote side closed connection
```

---

# Дуже важливо

recv() після disconnect:

```text
НЕ блокується
```

Він:
- миттєво повертає EOF

---

# Правильний pattern

```python
while True:

    data = conn.recv(1024)

    if not data:
        break
```

---

# Чому це важливо?

Якщо НЕ перевіряти EOF:

server може:
- крутити infinite loop
- навантажувати CPU
- працювати некоректно

# 8. Partial Messages та framing problem

TCP НЕ має поняття:

```text
message
```

Є лише:

```text
byte stream
```

---

# Проблема

Client:

```python
sendall(b"HelloWorld")
```

Server:

```python
recv(5)
```

отримає:

```python
b"Hello"
```

І решта bytes:
- залишаться у buffer.

---

# Framing Problem

TCP НЕ знає:
- де message починається
- де закінчується

Тому application protocol мусить:
- сам визначати boundaries.

---

# Як це роблять?

## Варіант 1 — delimiter

```text
Hello\n
World\n
```

---

## Варіант 2 — length prefix

```text
[SIZE][DATA]
```

---

## Варіант 3 — fixed-size protocol

```text
always 1024 bytes
```

---

# Чому це важливо?

Бо без framing:
- JSON може бути incomplete
- file upload може ламатися
- protocol стає unstable

# 9. Deadlocks та buffered I/O

Deadlock —
це коли дві сторони:
- чекають одна одну
- і ніхто не рухається далі.

---

# Простий приклад

Client:

```python
send()
recv()
```

Server:

```python
recv()
send()
```

Все ок.

---

# Але buffered I/O може зламати це

Наприклад:

```python
file.write("Hello")
```

НЕ гарантує:
- що bytes одразу пішли у network.

Вони можуть:
- залишитися у Python buffer.

---

# І тоді:

Server:
- чекає recv()

Client:
- чекає response

І обидва:
- asleep forever

---

# Solution

```python
file.flush()
```

або:
- unbuffered sockets
- explicit protocol design

# 10. Threading та I/O concurrency

Головна проблема networking:

```text
recv() блокує process
```

---

# Sequential Server Problem

```python
while True:

    conn, addr = accept()

    data = recv()

    process()
```

Якщо один client:
- думає 30 секунд

server:
- не обробляє інших clients

---

# Thread per client

Рішення:

```text
1 client = 1 thread
```

---

# Ідея

Поки:
- Thread A waiting recv()

CPU може:
- виконувати Thread B

---

# Чому threading працює для networking?

Під час blocking I/O:

```text
Python RELEASES GIL
```

І OS scheduler:
- запускає інші threads.

---

# Але threading теж має проблеми

Багато threads:
- memory overhead
- context switching
- synchronization complexity

---

# Тому з’явився asyncio

Ідея asyncio:

```text
НЕ створювати thread на кожен socket
```

А:
- один event loop
- багато non-blocking sockets

In [ ]:
import socket
import threading
import time

# ======================================================
# ПОМИЛКА 1: recv() повертає b'' — але цикл не зупиняється
# ======================================================

def demo_empty_recv():
    """
    НЕПРАВИЛЬНО: цикл не перевіряє EOF → 100% CPU
    """
    PORT = 5070
    received = []

    def server():
        s = socket.socket()
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind(("127.0.0.1", PORT))
        s.listen(1)
        conn, _ = s.accept()

        count = 0
        while True:
            data = conn.recv(1024)
            count += 1
            if not data:
                print(f"  [ПРАВИЛЬНО] EOF detected після {count} ітерацій — виходимо")
                break
            received.append(data)
        conn.close()
        s.close()

    def client():
        time.sleep(0.3)
        c = socket.socket()
        c.connect(("127.0.0.1", PORT))
        c.sendall(b"Hello!")
        c.close()  # надсилає FIN → recv() поверне b''

    t1 = threading.Thread(target=server)
    t2 = threading.Thread(target=client)
    t1.start(); t2.start()
    t1.join(); t2.join()
    print(f"  Дані отримані: {received}")

print("=== DEMO: recv() EOF ===")
demo_empty_recv()

# ======================================================
# ПОМИЛКА 2: Сервер зависає назавжди — blocking accept()
# ======================================================
print("\n=== DEMO: Blocking accept() ===")

def demo_blocking_accept():
    s = socket.socket()
    s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    s.bind(("127.0.0.1", 5071))
    s.listen(1)
    s.settimeout(2.0)  # для демонстрації встановлюємо timeout
    print("  [SERVER] Чекаємо accept()... (timeout 2с)")
    try:
        conn, addr = s.accept()
    except socket.timeout:
        print("  [SERVER] Timeout! Нікого немає — але сервер не завис назавжди")
    s.close()

demo_blocking_accept()

# ======================================================
# ПОМИЛКА 3: Часткові повідомлення (partial messages)
# ======================================================
print("\n=== DEMO: Partial Messages ===")

def demo_partial():
    PORT = 5072

    def server():
        s = socket.socket()
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind(("127.0.0.1", PORT))
        s.listen(1)
        conn, _ = s.accept()

        # НЕПРАВИЛЬНО: recv(3) не гарантує повне слово
        chunk1 = conn.recv(3)
        chunk2 = conn.recv(1024)
        print(f"  [SERVER] Chunk 1: {chunk1}")
        print(f"  [SERVER] Chunk 2: {chunk2}")
        print(f"  [SERVER] Full message: {chunk1 + chunk2}")
        conn.close()
        s.close()

    def client():
        time.sleep(0.3)
        c = socket.socket()
        c.connect(("127.0.0.1", PORT))
        c.sendall(b"HelloWorld")
        c.close()

    t1 = threading.Thread(target=server)
    t2 = threading.Thread(target=client)
    t1.start(); t2.start()
    t1.join(); t2.join()

demo_partial()

# ======================================================
# ПОМИЛКА 4: Broken connection (ConnectionRefusedError)
# ======================================================
print("\n=== DEMO: Broken Connection ===")

def demo_connection_refused():
    c = socket.socket()
    c.settimeout(2.0)
    try:
        c.connect(("127.0.0.1", 9999))  # нічого не слухає
    except ConnectionRefusedError as e:
        print(f"  [CAUGHT] ConnectionRefusedError: {e}")
    except socket.timeout:
        print("  [CAUGHT] Timeout: хост не відповів")
    finally:
        c.close()

demo_connection_refused()

# ======================================================
# ПОМИЛКА 6: Port conflict та SO_REUSEADDR
# ======================================================
print("\n=== DEMO: Port Conflict & SO_REUSEADDR ===")

def demo_port_reuse():
    # БЕЗ SO_REUSEADDR
    s1 = socket.socket()
    s1.bind(("127.0.0.1", 5073))
    s1.listen(1)

    s2 = socket.socket()
    try:
        s2.bind(("127.0.0.1", 5073))  # той самий порт!
    except OSError as e:
        print(f"  [БЕЗ reuseaddr] OSError: {e}")

    s1.close()
    s2.close()

    # З SO_REUSEADDR
    time.sleep(0.1)
    s3 = socket.socket()
    s3.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    s3.bind(("127.0.0.1", 5073))
    s3.listen(1)
    print("  [З SO_REUSEADDR] Порт зайнятий знову — ОК!")
    s3.close()

demo_port_reuse()

# ======================================================
# ПОМИЛКА 8: Non-blocking socket і BlockingIOError
# ======================================================
print("\n=== DEMO: Non-blocking socket ===")

def demo_nonblocking():
    s = socket.socket()
    s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    s.bind(("127.0.0.1", 5074))
    s.listen(1)
    s.setblocking(False)
    try:
        conn, addr = s.accept()
    except BlockingIOError:
        print("  [CAUGHT] BlockingIOError: немає клієнтів прямо зараз")
        print("  [INFO] Non-blocking socket не чекає — він кричить 'не готово!'")
    s.close()

demo_nonblocking()

# ======================================================
# ПОМИЛКА 10: Bytes vs Strings
# ======================================================
print("\n=== DEMO: Bytes vs Strings ===")

def demo_encoding():
    PORT = 5075

    def server():
        s = socket.socket()
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind(("127.0.0.1", PORT))
        s.listen(1)
        conn, _ = s.accept()
        raw = conn.recv(1024)
        print(f"  [SERVER] raw bytes: {raw}")
        print(f"  [SERVER] decoded:   {raw.decode('utf-8')}")
        conn.close()
        s.close()

    def client():
        time.sleep(0.3)
        c = socket.socket()
        c.connect(("127.0.0.1", PORT))

        # НЕПРАВИЛЬНО: c.sendall("Привіт!")  → TypeError
        # ПРАВИЛЬНО:
        message = "Привіт, мережа!"
        c.sendall(message.encode("utf-8"))
        c.close()

    t1 = threading.Thread(target=server)
    t2 = threading.Thread(target=client)
    t1.start(); t2.start()
    t1.join(); t2.join()

demo_encoding()

print("\n=== Всі демо завершено ===")


# 15. П'ять питань-пасток: де ламається мислення початківців

Ці питання побудовані навколо найчастіших хибних уявлень.
Спробуй відповісти ДО того, як читати пояснення.

---

## Питання-пастка 1: Ілюзія "1024 bytes"

```python
# --- Клієнт ---
sock.sendall(b"User:")
sock.sendall(b"Alice")

# --- Сервер ---
data = client_sock.recv(1024)
print(data.decode())
```

**Що надрукує сервер?**

- A) `User:`
- B) `User:Alice`
- C) `Alice`
- D) Неможливо передбачити точно

**Правильна відповідь: D**

TCP — це потік байтів, НЕ черга повідомлень.
`recv(1024)` означає: "дай мені все, що зараз є в буфері, до 1024 байт".
Якщо мережа швидка — буде `b"User:Alice"`.
Якщо другий пакет ще не прийшов — буде тільки `b"User:"`.

**Аналогія:** Відро і водопровід. `sendall()` ллє воду. `recv()` зачерпує відром.
Відро може зачерпнути воду від одного, двох або половини вливань — залежно від моменту.

---

## Питання-пастка 2: Тихий Deadlock

```python
# --- Клієнт ---
file_wrapper = sock.makefile('w')
file_wrapper.write("Hello Server\n")   # дані в буфері Python, НЕ в мережі!

reply = sock.recv(1024)                # чекає відповіді

# --- Сервер ---
data = client_sock.recv(1024)          # чекає даних від клієнта
client_sock.sendall(b"Got it!")
```

**Що відбудеться?**

- A) Клієнт отримає `"Got it!"`
- B) Сервер впаде з помилкою
- C) Обидва зависнуть назавжди

**Правильна відповідь: C — Deadlock**

`makefile('w')` буферизує дані в пам'яті Python. `write()` НЕ надсилає нічого в мережу.
Клієнт чекає відповіді → Сервер чекає даних → ніхто не рухається.

**Рішення:** `file_wrapper.flush()` після `write()`.

**Аналогія:** Поштова скринька. `write()` кладе листа у вихідний лоток.
Листоноша забирає лоток лише коли він повний або ти викличеш `flush()`.
Клієнт стоїть біля дверей і чекає відповіді, хоча лист досі лежить на його столі.

---

## Питання-пастка 3: GIL і багатопоточність

```python
import threading, requests

def fetch():
    requests.get("http://example.com")  # 1 секунда

threads = [threading.Thread(target=fetch) for _ in range(100)]
for t in threads: t.start()
for t in threads: t.join()
```

**Скільки часу займе виконання? GIL не дозволяє паралельний код.**

- A) ~100 секунд (GIL змушує виконувати по черзі)
- B) ~1 секунда (threads паралельні)
- C) Впаде через GIL

**Правильна відповідь: B — ~1 секунда**

GIL звільняється під час blocking I/O. Поки thread чекає відповіді від сервера,
він відпускає GIL. Інші threads можуть виконуватись.

**Аналогія:** 100 кухарів з одним ножем (GIL).
Якщо ріжуть цибулю (CPU) — 99 чекають.
Якщо поставили піцу в мікрохвильовку (network I/O) — відклали ніж.
Всі 100 піц у мікрохвильовках одночасно!

---

## Питання-пастка 4: Привид у порту

```python
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.bind(('localhost', 8080))
server.listen(5)
# Натискаємо Ctrl+C, одразу перезапускаємо скрипт
```

**Що станеться при перезапуску?**

- A) Сервер знову слухає порт 8080
- B) `OSError: [Errno 98] Address already in use`
- C) Автоматично перемикається на порт 8081

**Правильна відповідь: B**

ОС тримає порт у стані `TIME_WAIT` після закриття з'єднання,
щоб "старі" пакети з мережі не потрапили у нову програму.

**Рішення:** `server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)` перед `bind()`.

**Аналогія:** Орендар з'їхав, але поштовий офіс ще 2 місяці пересилає пошту на стару адресу.
Новий орендар не може зареєструватись — поки хазяїн явно не скасує пересилання (`SO_REUSEADDR`).

---

## Питання-пастка 5: Вбивця Event Loop

```python
import asyncio, time

async def handle_client(reader, writer):
    print("Клієнт підключився!")
    time.sleep(5)   # УВАГА: звичайний sleep, не await!
    writer.write(b"Done")
    await writer.drain()

async def main():
    server = await asyncio.start_server(handle_client, '127.0.0.1', 8888)
    await server.serve_forever()

asyncio.run(main())
```

**10 клієнтів підключились одночасно. Скільки чекатиме останній?**

- A) ~5 секунд (всі паралельно)
- B) ~50 секунд (по черзі)
- C) Сервер впаде

**Правильна відповідь: B — ~50 секунд**

`asyncio` — це ONE thread з cooperative multitasking.
`time.sleep(5)` — це synchronous блокування ВСЬОГО thread.
Event loop замирає повністю. Жоден інший клієнт не може бути оброблений.

**Рішення:** `await asyncio.sleep(5)` — це async операція, яка повертає контроль event loop.

**Аналогія:** Один жонглер (event loop) підкидає кульки.
`await` = підкинув кульку і взяв наступну.
`time.sleep()` = закрив очі на 5 секунд. Всі інші кульки впали на підлогу.

# 16. Питання для роздумів

---

## Питання 1

Чому `accept()` повертає абсолютно НОВИЙ socket,
а не дозволяє клієнту писати напряму у listening socket?

> Підказка: що станеться з іншими клієнтами, якщо використовувати один socket?

---

## Питання 2

`sendall()` лише копіює дані у OS Send Buffer.

Як ти можеш бути впевнений, що дані РЕАЛЬНО дійшли до іншої машини?

> Підказка: що роблять ACK пакети у TCP?

---

## Питання 3

Чому мережеві бібліотеки вимагають `.encode()` перед відправкою?

```python
sock.sendall("Hello")        # TypeError
sock.sendall("Hello".encode())  # OK
```

> Підказка: що таке мережа з точки зору фізики?

---

## Питання 4

У asyncio-сервері є такий код:

```python
async def handle(reader, writer):
    import time
    time.sleep(10)   # не await!
    writer.write(b"Done")
```

10 клієнтів підключились одночасно.
Скільки часу чекатиме останній?

> Підказка: скільки threads у asyncio event loop?

---

## Питання 5

```python
sock.send(b"spam\n")
sock.send(b"eggs\n")
sock.send(b"ham\n")
```

Server:
```python
data = conn.recv(1024)
print(data)
```

Що надрукує server?
- A) `b'spam\n'`
- B) `b'spam\neggs\nham\n'`
- C) Неможливо передбачити точно

> Правильна відповідь: C. Через OS buffering результат залежить від швидкості мережі та scheduler'а.